## **1. Setup & Installation**

Install PySastrawi untuk pencarian kata dasar (stemming) Bahasa Indonesia.

In [18]:
!pip install PySastrawi langdetect
import pandas as pd
import numpy as np
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from langdetect import detect

## **2. Load Dataset**

Membaca file hasil scraping

In [19]:
df = pd.read_csv('ReviewAppDiscord.csv')
# Menghapus data kosong dan duplikat agar data berkualitas
df = df.dropna(subset=['content']).drop_duplicates(subset=['content'])

## **3. Language Filtering**

Memastikan ulasan benar-benar bahasa Indonesia

In [20]:
def is_indonesian(text):
    try: return detect(text) == 'id'
    except: return False

df = df[df['content'].apply(is_indonesian)]

## **4. Text Cleaning (Case Folding & Removal)**

Menghapus simbol, angka, dan mengubah ke huruf kecil.

In [21]:
def clean_text(text):
    text = text.lower() # Case folding
    text = re.sub(r'http\S+|www\S+|https\S+', '', text) # Hapus URL
    text = re.sub(r'[^\w\s]', ' ', text) # Hapus tanda baca
    text = re.sub(r'\d+', '', text) # Hapus angka
    text = re.sub(r'\s+', ' ', text).strip() # Hapus spasi berlebih
    return text

df['content_clean'] = df['content'].apply(clean_text)

## **5. Slang Word Normalization**

Mengubah kata gaul Indonesia menjadi kata baku (misal: "ga" -> "tidak").

In [22]:
slang_dict = {
    "ga": "tidak", "gak": "tidak", "gk": "tidak", "tdk": "tidak",
    "yg": "yang", "utk": "untuk", "bgt": "banget", "sdh": "sudah",
    "klo": "kalau", "kalo": "kalau", "dgn": "dengan", "aja": "saja"
}

def normalize_slang(text):
    words = text.split()
    normalized = [slang_dict.get(w, w) for w in words]
    return " ".join(normalized)

df['content_norm'] = df['content_clean'].apply(normalize_slang)

## **6. Stopwords Removal**

Menggunakan daftar kata buang bahasa Indonesia.

In [23]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

# Menggunakan stopwords Indonesia
list_stopwords = set(stopwords.words('indonesian'))
# Tambahkan custom stopwords jika ada kata yang sering muncul tapi tidak penting
list_stopwords.update(['discord', 'aplikasi', 'app', 'aplikasinya'])

df['content_stop'] = df['content_norm'].apply(lambda x: " ".join([w for w in x.split() if w not in list_stopwords]))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## **7. Stemming (Mencari Kata Dasar)**

Proses ini penting untuk Bahasa Indonesia agar kata seperti "memperbaiki" dan "perbaikan" menjadi satu kata dasar: "baik".

In [24]:
from tqdm.auto import tqdm # Menggunakan auto agar menyesuaikan dengan environment (Colab/Jupyter)
tqdm.pandas() # Menghubungkan tqdm dengan pandas

# Inisialisasi Stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

print("Memulai proses Stemming (Proses ini memakan waktu cukup lama)...")

# Gunakan progress_apply agar muncul bar loading-nya
df['content_stemmed'] = df['content_stop'].progress_apply(lambda x: stemmer.stem(str(x)))

# Tampilkan hasil sementara
display(df[['content_stop', 'content_stemmed']].head())

Memulai proses Stemming (Proses ini memakan waktu cukup lama)...


  0%|          | 0/11495 [00:00<?, ?it/s]

,content_stop,content_stemmed
0,sistem login ribet suka error suka error sih e...,sistem login ribet suka error suka error sih e...
1,menambah teman,tambah teman
2,bikin histori server join join,bikin histori server join join
3,ayolah tambahin fitur bikin nama email nya ng ...,ayo tambahin fitur bikin nama email nya ng ush...
5,bug mulu aneh gw masuk,bug mulu aneh gw masuk


## **8. Final Cleaning & Export**

Simpan hasilnya ke file csv

In [25]:
df_final = df[['score', 'at', 'content_stemmed']]
df_final.to_csv('ReviewAppDiscord_Clean.csv', index=False)
print("Preprocessing Selesai!")

Preprocessing Selesai!
